# StyleMe — 02: Linear Probes (only if a gate failed in 01)

A probe is a tiny linear head on top of the frozen MobileCLIP embedding:

```
logits = W · e + b        W: (classes × 512)
```

Why this and not fine-tuning:
- the embedding is already a strong generic feature; the weak part is the label geometry, which a linear head fixes;
- it ships as JSON and runs in the app's TypeScript as a dot product — no second model runtime;
- user tag corrections become training rows for the next probe version (the flywheel).

**Class coverage warning:** a probe can only predict the classes present in its training labels. The slot probe combines DeepFashion2 crops (4 slots) with Fashion Product images (all 6 slots), so it covers the full set. The color probe covers the 16-color palette. Do **not** probe `pattern` from the Fashionpedia subset — it has no graphic/animal examples, so zero-shot stays the default there.

Output: `artifacts/probes.json`, then re-bundles `styleme_vision_v1.zip`.

In [ ]:
!pip install -q scikit-learn

In [ ]:
import os, json, glob, subprocess
import numpy as np

ROOT = os.environ.get('STYLEME_ROOT')
if not ROOT:
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_notify=20)
        ROOT = '/content/drive/MyDrive/StyleMe'
    except Exception:
        ROOT = '/content/StyleMe'
ART = os.path.join(ROOT, 'artifacts')
print('ART:', ART)
print(sorted(os.listdir(ART)))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

data = np.load(os.path.join(ART, 'probe_data.npz'), allow_pickle=True)
report = json.load(open(os.path.join(ART, 'eval_report.json')))

# report key -> probe_data key
GATE_TO_GROUP = {'slot': 'slot', 'fit': 'fit', 'formality': 'formality', 'color_cv': 'color'}
failed = []
for gate_key, group in GATE_TO_GROUP.items():
    r = report.get(gate_key)
    if (isinstance(r, dict) and r.get('gate') is not None
            and r.get('acc') is not None and r['acc'] < r['gate']):
        failed.append(group)
print('failed gates that are probeable:', failed)

# Override manually if you want to force a probe:
PROBE_GROUPS = failed  # e.g. ['slot', 'color', 'formality', 'fit']
print('training probes for:', PROBE_GROUPS)

In [ ]:
probes = {}
for group in PROBE_GROUPS:
    X = data[f'X_{group}']
    y = data[f'y_{group}'].astype(str)
    if len(y) < 50:
        print(f'{group}: only {len(y)} rows — skip')
        continue
    counts = dict(zip(*np.unique(y, return_counts=True)))
    if min(counts.values()) < 5:
        print(f'{group}: class too rare {counts} — skip')
        continue

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=3000, C=1.0, multi_class='auto').fit(Xtr, ytr)
    acc = float(clf.score(Xte, yte))
    print(f'probe {group}: test acc {acc:.3f}  (n={len(y)}, classes={sorted(counts)})')

    probes[group] = {
        'labels': [str(c) for c in clf.classes_],
        'W': np.round(np.asarray(clf.coef_, dtype=np.float32), 5).tolist(),
        'b': np.round(np.asarray(clf.intercept_, dtype=np.float32), 5).tolist(),
        'acc': round(acc, 3),
        'n': int(len(y)),
    }

if not probes:
    print('nothing to probe — 01 gates either passed or need better labels')
else:
    json.dump(probes, open(os.path.join(ART, 'probes.json'), 'w'), indent=1)
    print('wrote probes.json:', {k: v['acc'] for k, v in probes.items()})

    # Fold probe accuracy into the shipped report so it reflects what runs.
    report_path = os.path.join(ART, 'eval_report.json')
    rep = json.load(open(report_path))
    key_for = {'color': 'color_cv'}
    for group, p in probes.items():
        key = key_for.get(group, group)
        if isinstance(rep.get(key), dict):
            rep[key]['probe_acc'] = p['acc']
            rep[key]['probe_n'] = p['n']
            rep[key]['shipped'] = 'probe'
    json.dump(rep, open(report_path, 'w'), indent=1)

    lines = [f"# StyleMe vision eval report", '', f"model: **{rep.get('model')}**", '']
    for name, r in rep.items():
        if not isinstance(r, dict):
            continue
        acc = r.get('acc')
        gate = r.get('gate')
        probe = r.get('probe_acc')
        if acc is None:
            lines.append(f'- **{name}**: skipped — {r.get("note")}')
        elif gate is None:
            lines.append(f'- **{name}**: {acc:.3f} (diagnostic, n={r["n"]})')
        else:
            ok = acc >= gate
            lines.append(f'- **{name}**: zero-shot {acc:.3f} vs gate {gate} — {"PASS" if ok else "FAIL"} (n={r["n"]})')
        if probe is not None:
            lines.append(f'  - probe: **{probe:.3f}** (n={r.get("probe_n")}) — shipped')
    lines += ['', 'Probes override zero-shot for slot/color/formality. Pattern/fit stay zero-shot until a train-split probe exists.']
    open(os.path.join(ART, 'eval_report.md'), 'w').write(chr(10).join(lines))
    print(chr(10).join(lines))

## Optional: fold in user corrections

The app exports corrections as JSONL rows:

```json
{"attr":"fit","value":"oversized","embedding":[0.01, ... 512 floats]}
```

Drop `corrections.jsonl` in `ROOT` and re-run the training cell above with these rows appended.

In [ ]:
corr_path = os.path.join(ROOT, 'corrections.jsonl')
corrections = []
if os.path.exists(corr_path):
    for line in open(corr_path):
        line = line.strip()
        if line:
            corrections.append(json.loads(line))
print(f'{len(corrections)} user corrections available')
if corrections:
    print('NOTE: re-run the training cell with these merged, then re-save probes.json.')
    print('      (merge snippet kept explicit so it cannot silently corrupt the probe.)')

## Re-bundle

Adds `probes.json` to the existing bundle. The app uses a probe only for its attribute group; everything else stays zero-shot.

In [ ]:
zip_path = os.path.join(ART, 'styleme_vision_v1.zip')
files = []
for name in ['prompt_embeddings.npy', 'taxonomy.json', 'eval_report.md', 'eval_report.json',
             'probe_data.npz', 'probes.json', 'visual_grid_slots.png']:
    p = os.path.join(ART, name)
    if os.path.exists(p):
        files.append(p)
files += glob.glob(os.path.join(ART, '*.tflite')) + glob.glob(os.path.join(ART, '*.onnx'))
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(['zip', '-q', '-j', zip_path] + files, check=True)
print('bundle:', zip_path, f'{os.path.getsize(zip_path)/1e6:.1f} MB')
print('contents:', [os.path.basename(f) for f in files])

## App wiring

In `apps/mobile/src/vision/`:

1. MobileCLIP TFLite/ONNX → 512-d embedding `e` (normalized).
2. cosine vs `prompt_embeddings` → zero-shot labels.
3. if `probes.json[group]` exists → `argmax(W·e + b)` over `labels` replaces that group's label.

`packages/engine` does not change — it consumes the final `GarmentAttrs`.